# Smart City Mission Vehicle — FINAL SIMPLE HANDHELD DEMO

這版只做一件事：**手拿卡牌/圖片給鏡頭看，車子依照固定順序反應。**

流程：

- 先辨識任務卡：`RED` 或 `GREEN`
- 如果是 `RED`：依序給它看 `fire → fire_extinguisher → fire_station`
  - `fire`：原地轉一圈
  - `fire_extinguisher`：原地轉一圈
  - `fire_station`：停止，任務結束
- 如果是 `GREEN`：依序給它看 `weed → weed_cutter → farm`
  - `weed`：原地轉一圈
  - `weed_cutter`：原地轉一圈
  - `farm`：停止，任務結束

這版已移除：找路、顏色入口、導航、直接物件模式、複雜搜尋狀態。


## 1. Settings
只需要優先改這裡的速度、時間、threshold。其他先不要動。


In [ ]:
from pathlib import Path
from enum import Enum
import os
import time

import cv2
import numpy as np
import torch
import torch.nn.functional as F
import torchvision

PROJECT_ROOT = Path.cwd()
MODEL_DIR = PROJECT_ROOT / "models"

# 如果只是用電腦測邏輯、不想讓車動，改 True。
DRY_RUN = False

# 通常先用 None。如果馬達完全不動，再試 "dfrobot" 或 "waveshare"。
ROBOT_DRIVER_BOARD = None

JETBOT_PASSWORD = "jetbot"
CAMERA_WIDTH = 224
CAMERA_HEIGHT = 224
CAMERA_FPS = 5

# 保留你們原本的模型檔名與資料夾位置。
TASK_CARD_MODEL_PATH = MODEL_DIR / "task_card_model.pth"
OBJECT_MODEL_PATH = MODEL_DIR / "object_model.pth"

# 馬達設定：手持展示只需要原地轉。
SPIN_SPEED = 0.28
SPIN_DURATION = 1.25

# 辨識設定：先求會動，不要設太高。
CARD_THRESHOLD = 0.65
OBJECT_THRESHOLD = 0.60
CARD_STABLE_COUNT = 1
OBJECT_STABLE_COUNT = 1

# RED/GREEN 卡牌：預設不用模型硬猜，改用文字形狀判斷。
# 原理：GREEN 這個字比較長，RED 比較短；它看的是字本身，不是字的顏色。
USE_TEXT_SHAPE_CARD_DETECTOR = True
USE_TASK_CARD_MODEL_FALLBACK = False
TEXT_CARD_ASPECT_THRESHOLD = 2.55
TEXT_CARD_MIN_AREA_RATIO = 0.006

# 如果你確定 task_card_model.pth 可用，而且想用模型判斷 RED/GREEN，
# 可以把 USE_TASK_CARD_MODEL_FALLBACK 改 True。
# 注意：ImageFolder 訓練通常是 ["GREEN", "RED"]，但你們舊程式是 ["RED", "GREEN"]。
TASK_CARD_CLASSES = ["RED", "GREEN"]
# TASK_CARD_CLASSES = ["GREEN", "RED"]  # 如果 debug 發現 RED/GREEN 顛倒，改成這行。

# 這裡保留你們原本 object_model 的順序，不亂改。
# 如果 debug 發現物件名稱顛倒，再改這裡，不要改任務流程。
OBJECT_CLASSES = [
    "fire",
    "fire_station",
    "fire_extinguisher",
    "farm",
    "weed_cutter",
    "weed",
]

# 如果你們的 object_model 是用 ImageFolder 重新訓練，通常順序會是下面這個：
# OBJECT_CLASSES = ["farm", "fire", "fire_extinguisher", "fire_station", "weed", "weed_cutter"]

TASK_CONFIG = {
    "RED": {
        "sequence": ["fire", "fire_extinguisher", "fire_station"],
        "actions": ["spin", "spin", "stop"],
    },
    "GREEN": {
        "sequence": ["weed", "weed_cutter", "farm"],
        "actions": ["spin", "spin", "stop"],
    },
}

print("Task card model:", TASK_CARD_MODEL_PATH)
print("Object model:", OBJECT_MODEL_PATH)
print("DRY_RUN:", DRY_RUN)
print("ROBOT_DRIVER_BOARD:", ROBOT_DRIVER_BOARD)
print("SPIN_SPEED / SPIN_DURATION:", SPIN_SPEED, SPIN_DURATION)
print("Card detector: text-shape =", USE_TEXT_SHAPE_CARD_DETECTOR, "/ model fallback =", USE_TASK_CARD_MODEL_FALLBACK)


## 2. Robot and safety helpers
保留 JetBot 初始化、停止、風扇、電源模式。


In [ ]:
class RobotController:
    def __init__(self, robot=None, dry_run=DRY_RUN, driver_board=ROBOT_DRIVER_BOARD):
        self.dry_run = dry_run
        self.robot = robot

        if self.robot is None and not self.dry_run:
            from jetbot import Robot
            try:
                if driver_board:
                    self.robot = Robot(driver_board=driver_board)
                else:
                    self.robot = Robot()
                print("Robot initialized.")
                self.hard_stop()
            except Exception as exc:
                print("Robot initialization failed:", exc)
                print("如果馬達完全不動，嘗試 ROBOT_DRIVER_BOARD = 'dfrobot' 或 'waveshare'，再 restart kernel。")
                raise

    def _run(self, name, speed):
        if self.dry_run:
            print(f"[DRY_RUN] {name}({speed})")
            return
        if self.robot is None:
            print("Robot is None; cannot move.")
            return
        getattr(self.robot, name)(speed)

    def spin_left(self):
        self._run("left", SPIN_SPEED)

    def stop(self):
        if self.dry_run:
            print("[DRY_RUN] stop")
            return
        if self.robot is None:
            return
        try:
            self.robot.stop()
        except Exception:
            pass
        try:
            self.robot.left_motor.value = 0
            self.robot.right_motor.value = 0
        except Exception:
            pass

    def hard_stop(self, repeat=3, delay=0.04):
        for _ in range(repeat):
            self.stop()
            time.sleep(delay)

    def spin_once(self):
        print("Action: spin once")
        if self.dry_run:
            print("[DRY_RUN] spin once")
            time.sleep(SPIN_DURATION)
            return
        try:
            self.spin_left()
            time.sleep(SPIN_DURATION)
        finally:
            self.hard_stop(repeat=4, delay=0.04)


def run_sudo_command(command, password=JETBOT_PASSWORD, dry_run=DRY_RUN):
    if dry_run:
        print(f"[DRY_RUN] {command}")
        return 0
    return os.system("echo %s | %s" % (password, command))


def set_10w_power_mode(dry_run=DRY_RUN):
    return run_sudo_command("sudo -S nvpmodel -m0", dry_run=dry_run)


def fan_on(dry_run=DRY_RUN):
    return run_sudo_command("sudo -S sh -c 'echo 255 > /sys/devices/pwm-fan/target_pwm'", dry_run=dry_run)


def fan_off(dry_run=DRY_RUN):
    return run_sudo_command("sudo -S sh -c 'echo 0 > /sys/devices/pwm-fan/target_pwm'", dry_run=dry_run)


robot = RobotController(dry_run=DRY_RUN, driver_board=ROBOT_DRIVER_BOARD)
robot.hard_stop()


## 2.1 Motor test
先手動跑一次。車子應該會原地轉一下，然後停止。


In [ ]:
# Motor test. Run manually before the mission.
robot.spin_once()
robot.hard_stop()
print("Motor test done.")


## 3. JetBot power and fan


In [ ]:
set_10w_power_mode(dry_run=DRY_RUN)
fan_on(dry_run=DRY_RUN)
print("Power mode set and fan on.")


## 4. Vision and model helpers
卡牌辨識會先看文字形狀；物件辨識使用你們訓練好的 object model。


In [ ]:
IMAGENET_MEAN = 255.0 * np.array([0.485, 0.456, 0.406])
IMAGENET_STDEV = 255.0 * np.array([0.229, 0.224, 0.225])


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def preprocess(frame):
    device = get_device()
    normalize = torchvision.transforms.Normalize(IMAGENET_MEAN, IMAGENET_STDEV)
    x = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    x = x.transpose((2, 0, 1))
    x = torch.from_numpy(x).float()
    x = normalize(x)
    x = x.to(device)
    return x[None, ...]


def load_alexnet_model(model_path, num_classes, required=True):
    if not Path(model_path).exists():
        message = f"Model file not found: {model_path}"
        if required:
            raise FileNotFoundError(message)
        print("WARNING:", message)
        return None

    device = get_device()
    model = torchvision.models.alexnet(pretrained=False)
    model.classifier[6] = torch.nn.Linear(model.classifier[6].in_features, num_classes)
    state = torch.load(str(model_path), map_location=device)
    model.load_state_dict(state)
    model = model.to(device)
    model.eval()
    print("Loaded model:", model_path)
    return model


def predict_class(model, frame, class_names, threshold):
    if model is None or frame is None:
        return "none", 0.0
    with torch.no_grad():
        logits = model(preprocess(frame))
        probs = F.softmax(logits, dim=1).flatten()
        confidence, index = torch.max(probs, dim=0)
    confidence = float(confidence)
    label = class_names[int(index)]
    if confidence < threshold:
        return "none", confidence
    return label, confidence


def predict_task_card_by_text_shape(frame, debug=False):
    """
    RED/GREEN 卡牌救命辨識：看字的形狀，不看字的顏色。
    GREEN 字比較長，RED 字比較短，所以用文字 bounding box 的長寬比判斷。
    """
    if frame is None:
        return "none", 0.0

    img = cv2.resize(frame, (224, 224))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 只看中間，避免桌面邊緣、手、背景干擾。
    roi = np.zeros(gray.shape, dtype=np.uint8)
    roi[22:202, 12:212] = 255

    # 抓深色字與彩色字。這不是 OCR，只是抓字的形狀。
    dark_text = gray < 155
    colored_text = (hsv[:, :, 1] > 70) & (hsv[:, :, 2] < 245)
    mask = ((dark_text | colored_text).astype(np.uint8) * 255)
    mask = cv2.bitwise_and(mask, roi)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=1)

    points = cv2.findNonZero(mask)
    if points is None:
        if debug:
            print("TEXT_SHAPE: no text-like pixels")
        return "none", 0.0

    x, y, w, h = cv2.boundingRect(points)
    area_ratio = cv2.countNonZero(mask) / float(mask.shape[0] * mask.shape[1])
    aspect = w / max(h, 1)

    if area_ratio < TEXT_CARD_MIN_AREA_RATIO or w < 35 or h < 16:
        if debug:
            print(f"TEXT_SHAPE rejected: area={area_ratio:.4f}, bbox={w}x{h}, aspect={aspect:.2f}")
        return "none", 0.0

    label = "GREEN" if aspect >= TEXT_CARD_ASPECT_THRESHOLD else "RED"
    confidence = min(0.99, 0.72 + 0.12 * abs(aspect - TEXT_CARD_ASPECT_THRESHOLD))

    if debug:
        print(f"TEXT_SHAPE: {label}, conf={confidence:.2f}, aspect={aspect:.2f}, area={area_ratio:.4f}, bbox={w}x{h}")
    return label, confidence


def predict_task_card(frame):
    # 1) 先用文字形狀判斷 RED/GREEN。
    if USE_TEXT_SHAPE_CARD_DETECTOR:
        label, conf = predict_task_card_by_text_shape(frame)
        if label in TASK_CONFIG and conf >= CARD_THRESHOLD:
            return label, conf, "text_shape"

    # 2) 可選：再用 task_card_model fallback。
    if USE_TASK_CARD_MODEL_FALLBACK and task_model is not None:
        label, conf = predict_class(task_model, frame, TASK_CARD_CLASSES, CARD_THRESHOLD)
        return label, conf, "model"

    return "none", 0.0, "none"


def predict_object(frame):
    label, conf = predict_class(object_model, frame, OBJECT_CLASSES, OBJECT_THRESHOLD)
    return label, conf


## 5. Load models
如果只用文字形狀辨識卡牌，task card model 可以不存在；但 object model 一定要存在。


In [ ]:
task_model = load_alexnet_model(
    TASK_CARD_MODEL_PATH,
    len(TASK_CARD_CLASSES),
    required=USE_TASK_CARD_MODEL_FALLBACK,
)
object_model = load_alexnet_model(
    OBJECT_MODEL_PATH,
    len(OBJECT_CLASSES),
    required=True,
)
print("Models ready.")


## 6. Camera and simple mission logic
這裡就是最簡狀態機：`WAIT_CARD → WAIT_OBJECT → DONE`。


In [ ]:
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg

camera = Camera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT, fps=CAMERA_FPS)
image = widgets.Image(format="jpeg", width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
camera_link = traitlets.dlink((camera, "value"), (image, "value"), transform=bgr8_to_jpeg)

state_label = widgets.Label(value="state: WAIT_CARD")
task_label = widgets.Label(value="task: none")
target_label = widgets.Label(value="target: none")
detected_label = widgets.Label(value="detected: none")
action_label = widgets.Label(value="action: stop")
log_box = widgets.Textarea(value="Ready. Press Start Mission.", rows=12, layout=widgets.Layout(width="560px"))

start_button = widgets.Button(description="Start Mission", button_style="success")
reset_button = widgets.Button(description="Reset", button_style="warning")
force_red_button = widgets.Button(description="Force RED", button_style="danger")
force_green_button = widgets.Button(description="Force GREEN", button_style="success")

class SimpleMission:
    def __init__(self, robot):
        self.robot = robot
        self.reset()

    def reset(self):
        self.robot.hard_stop()
        self.state = "WAIT_CARD"
        self.task = None
        self.sequence = []
        self.actions = []
        self.index = 0
        self.current_target = None
        self.detected = "none"
        self.confidence = 0.0
        self.action = "stop"
        self.stable_label = None
        self.stable_count = 0
        self.log = ["Reset. Show RED or GREEN card."]
        return self.snapshot()

    def force_task(self, task_name):
        task_name = task_name.upper().strip()
        if task_name not in TASK_CONFIG:
            self.log.append(f"Unknown task: {task_name}")
            return self.snapshot()
        self._set_task(task_name, source="manual button")
        return self.snapshot()

    def _set_task(self, task_name, source="card"):
        self.robot.hard_stop()
        self.task = task_name
        self.sequence = TASK_CONFIG[task_name]["sequence"]
        self.actions = TASK_CONFIG[task_name]["actions"]
        self.index = 0
        self.current_target = self.sequence[0]
        self.state = "WAIT_OBJECT"
        self.action = "stop"
        self.stable_label = None
        self.stable_count = 0
        self.log.append(f"Task confirmed: {task_name} ({source})")
        self.log.append(f"Now show: {self.current_target}")

    def _stable_seen(self, label, target, need_count):
        if label == target:
            if self.stable_label == label:
                self.stable_count += 1
            else:
                self.stable_label = label
                self.stable_count = 1
        else:
            self.stable_label = label
            self.stable_count = 0
        return self.stable_count >= need_count

    def _finish(self):
        self.robot.hard_stop()
        self.state = "DONE"
        self.action = "stop"
        self.log.append("Final target detected. Stop. Mission complete.")

    def _do_current_action(self):
        target = self.current_target
        action = self.actions[self.index]
        self.log.append(f"Confirmed target: {target}")

        if action == "spin":
            self.action = "spin_once"
            self.robot.spin_once()
            self.index += 1
            self.stable_label = None
            self.stable_count = 0
            if self.index >= len(self.sequence):
                self._finish()
            else:
                self.current_target = self.sequence[self.index]
                self.action = "stop"
                self.log.append(f"Next show: {self.current_target}")
        else:
            self._finish()

    def step(self, frame):
        # DONE 後保持停止，不再動。
        if self.state == "DONE":
            self.robot.hard_stop(repeat=1, delay=0.01)
            return self.snapshot()

        # 等任務卡：RED or GREEN。
        if self.state == "WAIT_CARD":
            self.robot.hard_stop(repeat=1, delay=0.01)
            self.action = "stop"
            card, conf, method = predict_task_card(frame)
            self.detected = f"card:{card} ({method})"
            self.confidence = conf
            if card in TASK_CONFIG and conf >= CARD_THRESHOLD:
                if self._stable_seen("CARD_" + card, "CARD_" + card, CARD_STABLE_COUNT):
                    self._set_task(card, source=method)
            return self.snapshot()

        # 等目前指定物件。
        if self.state == "WAIT_OBJECT":
            self.robot.hard_stop(repeat=1, delay=0.01)
            self.action = "stop_waiting"
            label, conf = predict_object(frame)
            self.detected = label
            self.confidence = conf

            # 重點：只認目前 target。不是目前 target 就不動。
            if label == self.current_target and conf >= OBJECT_THRESHOLD:
                if self._stable_seen("OBJ_" + label, "OBJ_" + self.current_target, OBJECT_STABLE_COUNT):
                    self._do_current_action()
            else:
                self.stable_label = label
                self.stable_count = 0
            return self.snapshot()

        return self.snapshot()

    def snapshot(self):
        return {
            "state": self.state,
            "task": self.task or "none",
            "target": self.current_target or "none",
            "detected": self.detected,
            "confidence": self.confidence,
            "action": self.action,
            "stable_label": self.stable_label,
            "stable_count": self.stable_count,
            "log": self.log[-30:],
        }

mission = SimpleMission(robot)


def refresh_ui(s):
    state_label.value = f"state: {s['state']}"
    task_label.value = f"task: {s['task']}"
    target_label.value = f"target: {s['target']}"
    detected_label.value = f"detected: {s['detected']} ({s['confidence']:.2f}) | stable: {s['stable_label']} x{s['stable_count']}"
    action_label.value = f"action: {s['action']}"
    log_box.value = "\n".join(s["log"])


def on_reset(_):
    refresh_ui(mission.reset())


def on_force_red(_):
    refresh_ui(mission.force_task("RED"))


def on_force_green(_):
    refresh_ui(mission.force_task("GREEN"))

reset_button.on_click(on_reset)
force_red_button.on_click(on_force_red)
force_green_button.on_click(on_force_green)

display(widgets.HBox([
    image,
    widgets.VBox([
        widgets.HBox([start_button, reset_button]),
        widgets.HBox([force_red_button, force_green_button]),
        state_label,
        task_label,
        target_label,
        detected_label,
        action_label,
        log_box,
    ])
]))

refresh_ui(mission.snapshot())


## 7. Debug current frame
如果它看錯圖片，先跑這格看目前鏡頭前面到底被判成什麼。


In [ ]:
def print_top_predictions(model, frame, class_names, title="model", topk=6):
    if model is None:
        print(title, "is None")
        return
    with torch.no_grad():
        probs = F.softmax(model(preprocess(frame)), dim=1).flatten()
    k = min(topk, len(class_names))
    values, indices = torch.topk(probs, k)
    print(title)
    for value, index in zip(values, indices):
        print(f"  {class_names[int(index)]}: {float(value):.3f}")


def debug_current_frame():
    frame = camera.value
    print("--- TEXT-SHAPE CARD DETECTOR ---")
    predict_task_card_by_text_shape(frame, debug=True)
    print("\n--- TASK CARD MODEL ---")
    print_top_predictions(task_model, frame, TASK_CARD_CLASSES, "TASK CARD MODEL", topk=2)
    print("\n--- OBJECT MODEL ---")
    print_top_predictions(object_model, frame, OBJECT_CLASSES, "OBJECT MODEL", topk=6)

# 手拿一張圖到鏡頭前，再取消下一行註解執行。
# debug_current_frame()


## 8. Start mission
按這格之後開始觀察相機。順序：先給 RED/GREEN 卡，再依照畫面 target 給下一張圖。


In [ ]:
_is_updating = False


def update(change):
    global _is_updating
    if _is_updating:
        return
    _is_updating = True
    try:
        frame = change["new"]
        snapshot = mission.step(frame)
        refresh_ui(snapshot)
    finally:
        _is_updating = False


def start_mission(_=None):
    try:
        camera.unobserve(update, names="value")
    except Exception:
        pass
    robot.hard_stop()
    refresh_ui(mission.reset())
    camera.observe(update, names="value")
    print("Mission started.")
    print("Step 1: show RED or GREEN card.")
    print("RED sequence: fire -> fire_extinguisher -> fire_station.")
    print("GREEN sequence: weed -> weed_cutter -> farm.")

start_button.on_click(start_mission)

# 直接執行這格也會開始，不一定要按按鈕。
start_mission()


## 9. Stop mission safely
結束或出問題就跑這格。


In [ ]:
try:
    camera.unobserve(update, names="value")
except Exception:
    pass
robot.hard_stop()
print("Mission stopped. Wheels forced to stop.")


## 10. Turn fan off
最後才跑。


In [ ]:
try:
    camera.unobserve(update, names="value")
except Exception:
    pass
robot.hard_stop()
fan_off(dry_run=DRY_RUN)
# camera_link.unlink()
print("Fan off. Done.")
